In [ ]:
# %%
import gc
import json
from datetime import datetime
from pathlib import Path

import polars as pl
from tqdm.auto import tqdm

INPUT_DIR = Path("data/preprocessed_dataset")
# blind = 10% каналов (слепой a), june = июнь 2026 всех каналов (слепой b)
OUTPUT_DIRS = {s: Path(f"data/final_{s}") for s in ["train", "val", "blind", "june"]}

ID_COL = "ид_канала_данных"
SENSOR_COL = "тип_датчика"
with open("data/incident_by_sensor.json", encoding="utf-8") as f:
    INCIDENT_BY_SENSOR: dict[str, str] = json.load(f)

TARGET = "тревожное"
TARGET_UPPER_BOUND = 48
RANDOM_STATE = 69
BLIND_SIZE, VAL_SIZE = 0.10, 0.10  # train 80%
CUTOFF = datetime(2026, 6, 1)
# GAS и INTRUSION шлют поток строк: прореживаем только train, по хэшу ид_события
TRAIN_ROW_FRACTION = {"GAS": 0.15, "INTRUSION": 0.5}

ROW_DT = pl.col("дата").dt.combine(pl.col("время"))


def make_target(lf: pl.LazyFrame, censor: bool) -> pl.LazyFrame:
    t = pl.col(TARGET)
    if censor:
        # train/val: тревоги из июня (слепой период) не видим — цензура на CUTOFF
        to_cut = (pl.lit(CUTOFF) - ROW_DT).dt.total_seconds() / 3600.0
        t = pl.when(t.is_not_null() & (t <= to_cut)).then(t).otherwise(None)
        bound = pl.min_horizontal(pl.lit(float(TARGET_UPPER_BOUND)), to_cut)
    else:
        bound = pl.lit(float(TARGET_UPPER_BOUND))
    return lf.with_columns(
        pl.when(t.is_not_null()).then(pl.min_horizontal(t, pl.lit(float(TARGET_UPPER_BOUND)))).otherwise(bound).alias("target_lower"),
        pl.when(t <= TARGET_UPPER_BOUND).then(t).otherwise(-1.0).alias("target_upper"),
    ).drop(TARGET)


def device_split(files: list[Path]) -> pl.DataFrame:
    """Внутри класса: каналы по id, shuffle seed 69 -> blind 10%, val 10%, train 80%."""
    ch = (
        pl.scan_parquet(files).select(ID_COL, SENSOR_COL).unique(ID_COL)
        .with_columns(pl.col(SENSOR_COL).replace_strict(INCIDENT_BY_SENSOR, default=None).alias("__incident"))
        .drop_nulls("__incident").sort(ID_COL).collect()
    )
    parts = []
    for _, g in ch.group_by("__incident", maintain_order=True):
        ids = g.select(ID_COL).sample(fraction=1.0, shuffle=True, seed=RANDOM_STATE)
        n = ids.height
        nb, nv = int(n * BLIND_SIZE), int(n * VAL_SIZE)
        parts.append(ids.with_columns(
            pl.when(pl.int_range(n) < nb).then(pl.lit("blind"))
            .when(pl.int_range(n) < nb + nv).then(pl.lit("val"))
            .otherwise(pl.lit("train")).alias("__split")))
    return pl.concat(parts)


for d in OUTPUT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
files = sorted(INPUT_DIR.glob("*.parquet"))
split = device_split(files)
split.write_parquet("data/device_split.parquet")
print(split.group_by("__split").len().sort("__split"))

keep = pl.col("ид_события").hash(RANDOM_STATE) % 10_000 < (
    pl.col(SENSOR_COL).replace_strict(INCIDENT_BY_SENSOR, default=None)
    .replace_strict(TRAIN_ROW_FRACTION, default=1.0) * 10_000
)
before = ROW_DT < pl.lit(CUTOFF)
for fp in tqdm(files, desc="Processing chunks"):
    lf = pl.scan_parquet(fp).join(split.lazy(), on=ID_COL, how="inner")
    s = pl.col("__split")
    make_target(lf.filter(before & (s == "train") & keep), True).drop("__split").sink_parquet(OUTPUT_DIRS["train"] / fp.name)
    make_target(lf.filter(before & (s == "val")), True).drop("__split").sink_parquet(OUTPUT_DIRS["val"] / fp.name)
    make_target(lf.filter(before & (s == "blind")), False).drop("__split").sink_parquet(OUTPUT_DIRS["blind"] / fp.name)
    make_target(lf.filter(~before), False).with_columns((s == "blind").alias("__blind_channel")).drop("__split").sink_parquet(OUTPUT_DIRS["june"] / fp.name)
    gc.collect()


In [ ]:
for s in ["train", "val", "blind", "june"]:
    print(s, pl.scan_parquet(f"data/final_{s}/*.parquet").select(
        pl.len(), pl.col("ид_канала_данных").n_unique(),
        ((pl.col("target_upper") != -1) & (pl.col("target_lower") <= 24)).sum().alias("pos24")).collect())
